# Man-in-the-Middle — Selective GCS Spoofing

This vehicle is monitored by **two** GCS: `GCS_🔵` (the owner — the one that
actually launches and controls the vehicle's processes) and `GCS_🟠` (a second
station just watching the same telemetry). An attacker has man-in-the-middled
the link and, once the drone reaches a chosen waypoint
(`MISSION_CURRENT.seq >= trigger_seq`), injects one fabricated
`GLOBAL_POSITION_INT` — spoofed to look like it came from the vehicle itself —
toward `GCS_🟠` only, selected via a bitmask (`target_mask`, bit *i* = GCS at
position *i* in the vehicle's GCS list; bit 0 is the owner).

The real telemetry keeps flowing to **both** stations — the downlink relay
fans the same stream out to everyone — so this doesn't hide the true track,
it plants a fabricated report alongside it. `GCS_🔵`'s recorded trajectory
stays clean; `GCS_🟠`'s recorded trajectory picks up one bogus point far from
the real mission path.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.runtime.mitm import SpoofGCSStrategy
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)
speed = 5.0  # m/s
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_100,
#              seq=3 flying to north_200 ← attacker spoofs GCS_🟠 here
home_wp = ENU(x=0, y=0, z=0)
north_100 = ENU(x=0, y=100, z=cruise_alt)
north_200 = ENU(x=0, y=200, z=cruise_alt)
mission_wps = [home_wp, north_100, north_200]

# Fabricated position the attacker reports to GCS_🟠: 150 m SOUTH of origin,
# well off the real (northbound) mission path so the spoof is unmistakable.
spoof_target = gra_origin.unpose().to_abs(ENU(x=0, y=-150, z=cruise_alt))
print(f"Spoof target: lat={spoof_target.lat:.7f}, lon={spoof_target.lon:.7f}")

## Vehicle, watched by two GCS

In [ ]:
mission_path = "simulator/planner/missions/mitm_north.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=mission_path,
    navigation_speed=speed,
    firmware=model.firmware,
)

# Index 0 owns the vehicle (launches its processes); index 1 only monitors.
# Both record their own view so the spoof is visible in the comparison below.
gcs_owner = SimGCS(name=f"GCS_{Color.BLUE.emoji}", record_positions=True)
gcs_secondary = SimGCS(name=f"GCS_{Color.ORANGE.emoji}", record_positions=True)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    gcss=[gcs_owner, gcs_secondary],
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Simulator + MITM selective spoof

In [ ]:
orac = Oracle()

orac.add_vehicle(vehicle)

# `target_mask` bit i selects the GCS at position i in `vehicle.gcss`.
# 0b10 = bit 1 only → GCS_🟠 (index 1); GCS_🔵 (index 0, the owner) is left alone.
vehicle.mitm = SpoofGCSStrategy(
    trigger_seq=3,
    target_mask=0b10,
    spoof_lat=spoof_target.lat,
    spoof_lon=spoof_target.lon,
    spoof_alt=cruise_alt,
)

simulator = Simulator(oracle=orac, visualizer=gaz, verbose=1)

simulator.preview()

In [ ]:
simulator.run()

## Verifying the spoof landed on the right GCS only

- `simulator/logs/mitm/mitm_1.log` — `MITM spoof: reporting fake position ... to
  GCS [1]` (index 1 = `GCS_🟠`).
- The drone itself flies its real north mission throughout — this attack never
  touches Logic or the flight controller, only what the GCS is told.
- Compare the two stations' own recorded view below: `GCS_🔵` should show a
  clean north track; `GCS_🟠` should show the same track plus one bogus point
  far to the south.

In [ ]:
orac.plot_trajectories(oracle=False, gcss=[gcs_owner.name]);

In [ ]:
orac.plot_trajectories(oracle=False, gcss=[gcs_secondary.name]);